# ShiroFOC computational companion
Use the offline HTML lab for sliders and docs/NOTEBOOK.md for the guided explanation. This notebook exposes the reproducible Python model. Example motor and parasitic values are assumptions, not measurements. Run from this learning directory or the repository root.

In [ ]:
from pathlib import Path
import sys, json
root = Path.cwd() if Path('models/build_models.py').exists() else Path.cwd()/'learning'
sys.path.insert(0, str(root/'models'))
import build_models as model
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, SVG
print('Learning directory:', root)

## 1. Inputs
Predict which inputs are measured, design intent or assumptions before opening the file.

In [ ]:
inputs = json.loads((root/'data/board_inputs.json').read_text())
inputs['assumptions']

## 2. One half bridge
Change the winding L and PWM frequency. E is adjusted to hold the selected mean current; this is not a speed transient.

In [ ]:
t, voltage, winding, bridge, emf = model.half_bridge(v=36, d=0.5, fsw=20000, l=100e-6, r=0.1, mean_i=20)
plt.figure(figsize=(9,4))
plt.plot(t*1e6, winding, label='Winding')
plt.plot(t*1e6, bridge, label='Bridge bus draw')
plt.xlabel('Time (us)'); plt.ylabel('Current (A)'); plt.legend(); plt.show()

## 3. Impedance and current sharing
Change number of ceramics, retention and connection inductance. Are the dominant frequencies still away from the switching excitation?

In [ ]:
branches = [model.ceramic(40, retention=0.5, connection_nh=3)]
f = np.logspace(1,8,1500)
z, zs, zc = model.network(f, branches, rs=0.05, ls=1e-6)
plt.figure(figsize=(9,4)); plt.loglog(f, abs(z))
plt.xlabel('Frequency (Hz)'); plt.ylabel('Bus impedance (ohm)'); plt.show()

## 4. Three-phase periodic response
The phase currents are prescribed sinusoids. Bus voltage does not feed back into them; high-ripple cases exceed the model’s useful precision.

In [ ]:
t, v, load, source, caps, metrics = model.periodic_result(branches)
print(json.dumps(metrics, indent=2))
plt.figure(figsize=(9,4)); plt.plot(t*1e3, v)
plt.xlabel('Time (ms)'); plt.ylabel('Bus voltage (V)'); plt.show()

## 5. Load step and uncharged connection
Compare initial conditions. No clamps, precharge or current limits are included.

In [ ]:
for startup in [False, True]:
    t, out = model.rlc_response(c=94e-6, rc=0.0015, rs=0.05, ls=1e-6, startup=startup)
    plt.plot(t*1e3, out[:,0], label='Connection' if startup else 'Load step')
plt.xlabel('Time (ms)'); plt.ylabel('Bus voltage (V)'); plt.legend(); plt.show()

## 6. Energy and delay
Replace these values with your own example, then explain why energy buffering and measurement filtering are different design problems.

In [ ]:
C = 94e-6
print('Extra energy from 42 to 46 V:', 0.5*C*(46**2-42**2), 'J')
f_eval, tau, delay = 2000, 10e-6, 25e-6
print('Filter lag:', -np.degrees(np.arctan(2*np.pi*f_eval*tau)), 'deg')
print('Delay lag:', -360*f_eval*delay, 'deg')

## 7. Rebuild and inspect validation
This writes learning figures/data only. See qa/VALIDATION.md for physical limitations and the unsuccessful SPICE attempt.

In [ ]:
model.build()
json.loads((root/'qa/numerical_checks.json').read_text())

## 8. Your decision
Record an operating envelope, voltage budget, capacitor assumptions, placement rationale and the two measurements most likely to change your answer. Use docs/DECISION_RECORD.md as a worked example.